<a href="https://colab.research.google.com/github/SinghVipulRaj/basic-utility-tools/blob/test_files/yt_dl/audio_music.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install -y -qq ffmpeg
!pip install -q pydub tqdm yt-dlp

print("✅ Setup complete!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.5 MB/s eta 0:00:00
✅ Setup complete!


In [2]:
# import sys
# !{sys.executable} -m pip install yt-dlp
# !{sys.executable} -m pip install -U yt-dlp pydub scipy numpy tqdm
# # -------------------------------------------------------------------------------------------------------

# import subprocess

# # Install nodejs for yt-dlp to use as a JavaScript runtime
# print("Installing nodejs...")
# subprocess.run(["sudo", "apt-get", "update"], capture_output=True, text=True)
# subprocess.run(["sudo", "apt-get", "install", "-y", "nodejs"], capture_output=True, text=True)
# print("nodejs installed.")



In [3]:
# import yt_dlp
# import os
# import subprocess
# from google.colab import files
# import time

# # Changed to a Colab-friendly path
# base_output_dir="/content/yt_dwnlod"

# def download_video(urls, output_dir):
#     os.makedirs(output_dir, exist_ok=True)
#     ydl_opts = {
#         'format': 'bestaudio/best',
#         'outtmpl': os.path.join(output_dir, '%(title)s.%(ext)s'),
#         'ignoreerrors': True,
#     }
#     with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#         for url in urls:
#             try:
#                 info = ydl.extract_info(url, download=False, process=False)
#                 if "entries" in info:
#                     print(f"📜 Playlist found: {url} — {len(info['entries'])} items")
#                     for entry in info['entries']:
#                         if not entry:
#                             continue
#                         print(f"⬇️  Downloading: {entry['title']}")
#                         ydl.download([entry['webpage_url']])
#                 else:
#                     print(f"⬇️  Downloading: {info['title']}")
#                     ydl.download([url])
#             except Exception as e:
#                 print(f"❌  Failed to process URL: {url}\nReason: {e}")

# def convert_to_mp3(directory):
#     converted_files = []
#     for filename in os.listdir(directory):
#         if filename.endswith(".webm") or filename.endswith(".mp4"):
#             input_path = os.path.join(directory, filename)
#             base_name = os.path.splitext(filename)[0]
#             output_path = os.path.join(directory, base_name + ".mp3")

#             print(f"🎵 Converting: {filename} → {base_name}.mp3")
#             subprocess.run([
#                 "ffmpeg", "-i", input_path, "-vn", "-ab", "192k", "-ar", "44100", "-y", output_path
#             ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
#             if os.path.exists(input_path):
#                 os.remove(input_path)
#             converted_files.append(output_path)
#     return converted_files

# def runxert():
#     folder_name = input("Enter folder name (optional): ").strip()
#     output_dir = os.path.join(base_output_dir, folder_name) if folder_name else base_output_dir

#     # Example URL list - in a real scenario, you might want to prompt for this

#     # with open("yt_dwnlod/urls.txt", "r") as f:
#     #     urls = [line.strip() for line in f if line.strip()]

#     urls = [
#         # "https://www.youtube.com/watch?v=8ffErcweyIY"
#         # "https://youtu.be/8of5w7RgcTc?si=H0aQNIQfok1RnfkG"
#         # "https://www.youtube.com/watch?v=HTeP7ja9UFY&list=RDHTeP7ja9UFY&index=1&ab_channel=seventyskye",
#         # "https://www.youtube.com/watch?v=yMiIrAxQhFA&list=PL5jD2fLvy_Gpb4kh1hSg5gKS5qFndSsyP&ab_channel=SidewalksandSkeletons-Topic",
#         # "https://www.youtube.com/watch?v=q2u6Hr52Lno&ab_channel=medicomkvlog1206",
#         # "https://www.youtube.com/watch?v=j7TM2ccOGbU&list=PLUOEf-vLOCSkxWY5z9cjS4OT3oZ9D8suk&ab_channel=SuperHitGaane",
#         # "https://www.youtube.com/watch?v=ls5l5uNDfnU&list=PL5jD2fLvy_Gqx5jY9L1Q1n7GgCWyr8YpQ&ab_channel=MusicAcapellaForAll",
#     ]

#     download_video(urls, output_dir)
#     converted_mp3_paths = convert_to_mp3(output_dir)

#     print("\nInitiating downloads for converted MP3s...")
#     for path in converted_mp3_paths:
#         if os.path.exists(path):
#             files.download(path)
#     time.sleep(2)
#     print("converted MP3 should now be downloaded.")

In [4]:
import yt_dlp
import os
import subprocess
from google.colab import files
import time

# Changed to a Colab-friendly path
base_output_dir="/content/yt_dwnlod"

def download_file(urls, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    downloaded_paths = []

    # Intercept the path right after FFmpeg finishes audio conversion
    def hook(d):
        if d['status'] == 'finished' and d.get('postprocessor') == 'ExtractAudio':
            # ExtractAudio stores the converted file path in the info dictionary
            downloaded_paths.append(d['info_dict']['filepath'])

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': os.path.join(output_dir, '%(title)s.%(ext)s'),
        'ignoreerrors': True,
        'postprocessor_hooks': [hook],
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
    }
    current_batch_paths = []

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for url in urls:
            try:
                # Downloads and post-processes the audio, then returns metadata
                info = ydl.extract_info(url, download=True)
                if not info:
                    continue

                # Case 1: Playlist URL
                if 'entries' in info:
                    for entry in info['entries']:
                        if entry:
                            base_filename = ydl.prepare_filename(entry)
                            mp3_path = os.path.splitext(base_filename)[0] + '.mp3'
                            current_batch_paths.append(mp3_path)
                # Case 2: Single Video URL
                else:
                    base_filename = ydl.prepare_filename(info)
                    mp3_path = os.path.splitext(base_filename)[0] + '.mp3'
                    current_batch_paths.append(mp3_path)

            except Exception as e:
                print(f"❌ Failed to process {url}: {e}")

    return current_batch_paths

def run_download(urls, folder_name="new_downld"):
    output_dir = os.path.join(base_output_dir, folder_name) if folder_name else base_output_dir

    print("\n--Process STARTED--")
    paths = download_file(urls, output_dir)
    print("--Process COMPLETED--\n")
    for path in paths:
        if os.path.exists(path):
            files.download(path)
    time.sleep(2)
    print("MP3 can now be downloaded.")

In [5]:
import numpy as np
from pydub import AudioSegment
from scipy.signal import lfilter

def _load_audio(input_path):
    """Loads the audio file."""
    return AudioSegment.from_file(input_path)

def _apply_low_pass_filter(sound):
    """Applies a low-pass filter to simulate distance."""
    return sound.low_pass_filter(8000)

def _apply_reverb(left_samples, right_samples, sample_rate):
    """Applies concert hall reverb using a feedback delay line."""
    delay_seconds = 0.045  # 45ms delay for a large room feel
    delay_samples = int(sample_rate * delay_seconds)
    decay_factor = 0.35    # How fast the stadium echo dies out

    b = [1.0]
    a = [1.0] + [0.0] * (delay_samples - 1) + [-decay_factor]

    left_hall = lfilter(b, a, left_samples)
    right_hall = lfilter(b, a, right_samples)
    return left_hall, right_hall

def _apply_haas_effect(left_hall, right_hall, sample_rate):
    """Haas effect for speaker placement illusion."""
    haas_delay_samples = int(sample_rate * 0.020)
    right_hall_shifted = np.roll(right_hall, haas_delay_samples)
    right_hall_shifted[:haas_delay_samples] = 0  # Zero out the initial shift gap
    return left_hall, right_hall_shifted

def _rebuild_stereo_and_export(left_hall, right_hall, sample_rate, output_path):
    """Rebuilds stereo channels and exports the final audio."""
    left_hall_int = np.clip(left_hall, -32768, 32767).astype(np.int16)
    right_hall_int = np.clip(right_hall, -32768, 32767).astype(np.int16)

    processed_left = AudioSegment(left_hall_int.tobytes(), frame_rate=sample_rate, sample_width=2, channels=1)
    processed_right = AudioSegment(right_hall_int.tobytes(), frame_rate=sample_rate, sample_width=2, channels=1)

    concert_stereo = AudioSegment.from_mono_audiosegments(processed_left, processed_right)
    concert_stereo.export(output_path, format="mp3", bitrate="320k")

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [6]:
from tqdm.auto import tqdm
from scipy.signal import lfilter
def apply_concert_effect(input_path, output_path):
    stages = [
        "Loading audio",
        "Applying low-pass filter",
        "Processing channels (Reverb)",
        "Applying Haas effect",
        "Rebuilding stereo and Exporting audio"
    ]

    with tqdm(total=len(stages), desc="Applying Concert Effect") as pbar:
        # 1. Load the original audio file
        sound = _load_audio(input_path)
        pbar.update(1)
        pbar.set_postfix_str(stages[0] + " complete")

        # 2. Distance Logic: Apply a low-pass filter
        sound = _apply_low_pass_filter(sound)
        pbar.update(1)
        pbar.set_postfix_str(stages[1] + " complete")

        # Split the audio into raw arrays for digital signal processing
        left_channel, right_channel = sound.split_to_mono()
        left_samples = np.array(left_channel.get_array_of_samples(), dtype=np.float32)
        right_samples = np.array(right_channel.get_array_of_samples(), dtype=np.float32)
        sample_rate = sound.frame_rate

        # 3. Venue Reflection Logic: Create micro-delays
        left_hall, right_hall = _apply_reverb(left_samples, right_samples, sample_rate)
        pbar.update(1)
        pbar.set_postfix_str(stages[2] + " complete")

        # 4. Speaker Placement Logic: The Haas Effect
        left_hall_processed, right_hall_processed = _apply_haas_effect(left_hall, right_hall, sample_rate)
        pbar.update(1)
        pbar.set_postfix_str(stages[3] + " complete")

        # 5. Rebuild stereo and export
        _rebuild_stereo_and_export(left_hall_processed, right_hall_processed, sample_rate, output_path)
        pbar.update(1)
        pbar.set_postfix_str(stages[4] + " complete")
        print("Concert environment processing complete!")

In [7]:
def run_concert_effect_process():
    # --- User Interaction for File Selection and Loop ---
    while True:
        print("\n--- Ready for new audio file ---")
        print("Please upload MP3 file(s) to apply the concert effect.")
        uploaded = files.upload()

        if uploaded:
            processed_any = False
            for input_audio_filename in uploaded.keys():
                print(f"Uploaded file: {input_audio_filename}")

                # Automatically generate output filename
                base_filename = os.path.splitext(input_audio_filename)[0]
                output_audio_filename = f"concert_{base_filename}.mp3"
                print(f"Processed audio will be saved as: {output_audio_filename}")

                try:
                    # Run the concert effect script with user-provided files
                    apply_concert_effect(input_audio_filename, output_audio_filename)

                    # Provide download button for the converted audio
                    print(f"\nDownloading '{output_audio_filename}'...")
                    files.download(output_audio_filename)
                    print("Download initiated!")
                    processed_any = True
                except Exception as e:
                    print(f"Error processing {input_audio_filename}: {e}")

            if processed_any:
                print("Terminating. Thank you!")
            else:
                print("No files were successfully processed. Terminating.")
            break
        else:
            print("No file uploaded. Terminating.")
            break

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def main_menu():
    # Primary output container to display menus/inputs
    menu_output = widgets.Output()
    # Secondary output container to display task logs, downloads, and progress
    log_output = widgets.Output()

    # Define Main Buttons
    btn_download = widgets.Button(
        description="1. Download YouTube Song",
        button_style="primary",
        layout=widgets.Layout(width='250px', margin='10px')
    )

    btn_concert = widgets.Button(
        description="2. Apply Concert Effect",
        button_style="success",
        layout=widgets.Layout(width='250px', margin='10px')
    )

    def on_download_clicked(b):
        with menu_output:
            clear_output()
        with log_output:
            clear_output()

        with menu_output:
            print("\n--- YouTube Downloader ---")

            # Input for URLs
            url_input = widgets.Text(
                placeholder='Paste YouTube URL(s) here, comma-separated...',
                description='URLs:',
                layout=widgets.Layout(width='80%')
            )

            # Input for Folder Name
            folder_input = widgets.Text(
                value='new_downld',
                placeholder='Enter folder name (optional)...',
                description='Folder:',
                layout=widgets.Layout(width='80%')
            )

            submit_btn = widgets.Button(
                description="Start Download",
                button_style="info",
                layout=widgets.Layout(margin='10px 5px')
            )

            print("Fill in the details below and click 'Start Download' (or leave URLs blank to use defaults):")
            display(widgets.VBox([url_input, folder_input, submit_btn]))

            def on_submit(click_ev):
                # Clear ONLY the logs underneath while keeping buttons/fields visible
                with log_output:
                    clear_output()

                # Disable controls during download
                url_input.disabled = True
                folder_input.disabled = True
                submit_btn.disabled = True

                paste_choice = url_input.value.strip()
                chosen_folder = folder_input.value.strip() or "new_downld"

                # Execute processing and print logs inside log_output
                with log_output:
                    if paste_choice:
                        urls = [url.strip() for url in paste_choice.split(',') if url.strip()]
                    else:
                        urls = ["https://www.youtube.com/watch?v=hnRMZq1G6WU"]

                    print(f"\nProcessing URL(s): {urls}")
                    print(f"Saving into folder: {chosen_folder}")
                    run_download(urls, folder_name=chosen_folder)
                    print("--- YouTube Downloader finished ---\n")

                # Re-enable controls after completion
                url_input.disabled = False
                folder_input.disabled = False
                submit_btn.disabled = False

            submit_btn.on_click(on_submit)

    def on_concert_clicked(b):
        with menu_output:
            clear_output()
        with log_output:
            clear_output()

        with menu_output:
            print("\n--- Running Concert Effect Processor ---")

        with log_output:
            run_concert_effect_process()
            print("--- Concert Effect Processor finished ---")

    # Bind click events
    btn_download.on_click(on_download_clicked)
    btn_concert.on_click(on_concert_clicked)

    # Display Dashboard Title, Buttons, and Output Areas
    print("\n--- Welcome to the Audio Processing Dashboard ---")
    print("Please select an option below:")
    display(widgets.HBox([btn_download, btn_concert]))
    display(menu_output)
    display(log_output)

if __name__ == '__main__':
    main_menu()


--- Welcome to the Audio Processing Dashboard ---
Please select an option below:


Output()

Output()